[![Open In Colab](../../_static/colab-badge.svg)](https://colab.research.google.com/github/OpenProteinAI/openprotein-docs/blob/main/source/python-api/structure-generation/Using_RFdiffusion.ipynb)
[![Get Notebook](../../_static/get-notebook-badge.svg)](https://raw.githubusercontent.com/OpenProteinAI/openprotein-docs/refs/heads/main/source/python-api/structure-generation/Using_RFdiffusion.ipynb)
[![View In GitHub](../../_static/view-in-github-badge.svg)](https://github.com/OpenProteinAI/openprotein-docs/blob/main/source/python-api/structure-generation/Using_RFdiffusion.ipynb)

# Using BoltzGen
This tutorial shows you how to use the RFdiffusion model to design novel
protein structures.

The examples here are largely lifted from the [original
documentation](https://github.com/RosettaCommons/RFdiffusion) but
adapted to show how it can be run using the OpenProtein platform, which
can then be combined with our other workflows!

Full credit for the examples and use cases go to the authors of
RFdiffusion!

## Unconditional monomer design

The basic execution of RFdiffusion would be an unconditional design of a
protein structure of a certain length. You would need 2 things:

1.  Length of the protein
2.  Number of designs `N` desired

In [1]:
import openprotein
session = openprotein.connect()
length = 150
N = 3

In [2]:
boltzgen = session.models.boltzgen
boltzgen.generate?

Signature:
boltzgen.generate(
    design_spec: dict[str, typing.Any],
    structure_file: str | bytes | typing.BinaryIO | None = None,
    n: int = 1,
    **kwargs,
) -> openprotein.models.foundation.boltzgen.BoltzGenFuture
Docstring:
Run a protein structure generate job using BoltzGen.

Parameters
----------
design_spec : dict[str, Any]
    The BoltzGen design specification to run. This is the Python representation
    of the BoltzGen yaml request specification.
structure_file : BinaryIO, optional
    An input PDB file (as a file-like object) used for inpainting or other
    guided design tasks where parts of an existing structure are provided.
n : int, optional
    The number of unique design trajectories to run (default is 1).

Other Parameters
----------------
**kwargs : dict
    Additional keyword args that are passed directly to the boltzgen
    inference script. Overwrites any preceding options.

Returns
-------
BoltzGenFuture
    A future object that can be used to retrieve the

Craft the design specification, which is based on the [official YAML specification from BoltzGen](https://github.com/HannesStark/boltzgen).

In [3]:
design_spec = {
    "entities": [
        {
            "protein": {
                "id": "A",
                "sequence": str(length)
            }
        }
    ]
}

Run the design using BoltzGen:

In [4]:
design = boltzgen.generate(N=N, design_spec=design_spec)
design

BoltzGenJob(job_id='89a8e389-0818-4208-b077-cb80393adb9f', job_type='/models/boltzgen', status=<JobStatus.PENDING: 'PENDING'>, created_date=datetime.datetime(2025, 10, 30, 15, 55, 37, 405506, tzinfo=TzInfo(UTC)), start_date=None, end_date=None, prerequisite_job_id=None, progress_message=None, progress_counter=0, sequence_length=None)

Wait for the job to finish running with `wait_until_done`.

In [5]:
design.wait_until_done(verbose=True, timeout=600)

Waiting: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [04:33<00:00,  2.74s/it, status=SUCCESS]


True

Retrieve the PDB file of the design. Use the `replicate` param to
specify the 0-indexed design index to retrieve, in this case `0` to `2`.

In [6]:
from molviewspec import create_builder
result = design.get(replicate=0)
builder = create_builder()
structure = builder.download(url="mystructure.cif")\
    .parse(format="mmcif")\
    .model_structure()\
    .component()\
    .representation()\
    .color(color="blue")
builder.molstar_notebook(data={'mystructure.cif': result}, width=500, height=400)

<IPython.core.display.Javascript object>

## Vanilla Protein Binding

One of the basic examples in BoltzGen is to do a vanilla protein binding. We can retrieve the spec from the examples and the example target directly from their github.

Note that we need to pass in a structure file to run this example. In the spec (as shown below), it refers to a path `1g13.cif`. This is auto-patched within our system to refer to the passed `structure_file`. 

In [7]:
import requests
import yaml
import json

design_spec = yaml.safe_load(requests.get("https://raw.githubusercontent.com/HannesStark/boltzgen/refs/heads/main/example/vanilla_protein/1g13prot.yaml").text)
structure_file = requests.get("https://raw.githubusercontent.com/HannesStark/boltzgen/refs/heads/main/example/vanilla_protein/1g13.cif").text

print(json.dumps(design_spec, indent=4))
from molviewspec import create_builder
result = design.get(replicate=0)
builder = create_builder()
structure = builder.download(url="mystructure.cif")\
    .parse(format="mmcif")\
    .model_structure()\
    .component()\
    .representation()\
    .color(color="blue")
builder.molstar_notebook(data={'mystructure.cif': structure_file}, width=500, height=400)

{
    "entities": [
        {
            "protein": {
                "id": "C",
                "sequence": "80..140"
            }
        },
        {
            "file": {
                "path": "1g13.cif",
                "include": [
                    {
                        "chain": {
                            "id": "A"
                        }
                    }
                ]
            }
        }
    ]
}


<IPython.core.display.Javascript object>

Now we can run the example:

In [8]:
design = boltzgen.generate(
    structure_file=structure_file,
    design_spec=design_spec,
    N=1,
)
design

BoltzGenJob(job_id='8d73ba0a-44a7-4f59-8223-842ba9992141', job_type='/models/boltzgen', status=<JobStatus.PENDING: 'PENDING'>, created_date=datetime.datetime(2025, 10, 30, 17, 38, 49, 879407, tzinfo=TzInfo(UTC)), start_date=None, end_date=None, prerequisite_job_id=None, progress_message=None, progress_counter=0, sequence_length=None)

In [9]:
design.wait_until_done(verbose=True, timeout=600)

Waiting: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [01:58<00:00,  1.19s/it, status=SUCCESS]


True

In [10]:
from molviewspec import create_builder
result = design.get()
builder = create_builder()
structure = builder.download(url="mystructure.cif")\
    .parse(format="mmcif")\
    .model_structure()\
    .component()\
    .representation()\
    .color(color="blue")
builder.molstar_notebook(data={'mystructure.cif': result}, width=500, height=400)

<IPython.core.display.Javascript object>

### Vanilla Peptide with Target Binding Site

Let's run the other example which involves designing a peptide binder, by retrieving the specs from the official BoltzGen repo.

In [11]:
design_spec = yaml.safe_load(requests.get("https://raw.githubusercontent.com/HannesStark/boltzgen/refs/heads/main/example/vanilla_peptide_with_target_binding_site/beetletert.yaml").text)
structure_file = requests.get("https://raw.githubusercontent.com/HannesStark/boltzgen/refs/heads/main/example/vanilla_peptide_with_target_binding_site/5cqg.cif").text

print(json.dumps(design_spec, indent=4))
from molviewspec import create_builder
result = design.get(replicate=0)
builder = create_builder()
structure = builder.download(url="mystructure.cif")\
    .parse(format="mmcif")\
    .model_structure()\
    .component()\
    .representation()\
    .color(color="blue")
builder.molstar_notebook(data={'mystructure.cif': structure_file}, width=500, height=400)

{
    "entities": [
        {
            "protein": {
                "id": "G",
                "sequence": "12..20"
            }
        },
        {
            "file": {
                "path": "5cqg.cif",
                "include": [
                    {
                        "chain": {
                            "id": "A"
                        }
                    }
                ],
                "binding_types": [
                    {
                        "chain": {
                            "id": "A",
                            "binding": "343,344,251"
                        }
                    }
                ],
                "structure_groups": "all"
            }
        }
    ]
}


<IPython.core.display.Javascript object>

In [12]:
design = boltzgen.generate(
    structure_file=structure_file,
    design_spec=design_spec,
    N=1,
)
design

/home/jmage/Projects/openprotein/openprotein-python-private/openprotein/base.py:136: UserWarning: The requested payload is >1MB. There might be some delays or issues in processing. If the request fails, please try again with smaller sizes.
  warnings.warn(


BoltzGenJob(job_id='5a87d18e-e087-4e3f-9aa6-ab50e767f5a9', job_type='/models/boltzgen', status=<JobStatus.PENDING: 'PENDING'>, created_date=datetime.datetime(2025, 10, 30, 17, 58, 29, 302761, tzinfo=TzInfo(UTC)), start_date=None, end_date=None, prerequisite_job_id=None, progress_message=None, progress_counter=0, sequence_length=None)

In [13]:
from molviewspec import create_builder
design.wait_until_done(verbose=True, timeout=600)
result = design.get()
builder = create_builder()
structure = builder.download(url="mystructure.cif")\
    .parse(format="mmcif")\
    .model_structure()\
    .component()\
    .representation()\
    .color(color="blue")
builder.molstar_notebook(data={'mystructure.cif': result}, width=500, height=400)

Waiting: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [01:18<00:00,  1.28it/s, status=SUCCESS]


<IPython.core.display.Javascript object>

# Next Steps

You can run more of the examples from the BoltzGen repository. Take note that any command line arguments to `boltzgen run` can be passed as `kwargs` to the `boltzgen.design` function.

You can also move on to the next step of the design pipeline by running inverse folding using PoET-2. Refer to the walkthrough of [Inverse Folding with PoET-2](../../walkthroughs/PoET-2_inverse_folding.ipynb) for an example.

In [14]:
result = design.wait(verbose=True, timeout=600)
# show only the first 10 lines
print("\n".join(result.splitlines()[:10]))

Waiting: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:38<00:00,  2.58it/s, status=SUCCESS]


ATOM      1  N   GLY A   1     -20.970  12.196  12.412  1.00  0.00
ATOM      2  CA  GLY A   1     -21.400  10.976  13.086  1.00  0.00
ATOM      3  C   GLY A   1     -20.229  10.027  13.306  1.00  0.00
ATOM      4  O   GLY A   1     -19.292   9.986  12.510  1.00  0.00
ATOM      5  N   GLY A   2     -20.221   9.300  14.400  1.00  0.00
ATOM      6  CA  GLY A   2     -19.185   8.320  14.703  1.00  0.00
ATOM      7  C   GLY A   2     -19.053   7.294  13.585  1.00  0.00
ATOM      8  O   GLY A   2     -17.949   6.860  13.255  1.00  0.00
ATOM      9  N   GLY A   3     -20.200   6.833  13.049  1.00  0.00
ATOM     10  CA  GLY A   3     -20.176   5.891  11.936  1.00  0.00
